# Pitcher IP Distribution Baseline

Compare IP distributions across steamer projections, playing-time model, and actuals for 2019-2024.
Define target ranges for phase 3 calibration.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fantasy_baseball_manager.db.connection import create_connection

# The DB lives in the main repo's data/ directory; ../data/ works from the
# notebooks/ folder of the main checkout but not from a git worktree.
# Fall back to the absolute path when the relative one doesn't exist.
from pathlib import Path

_rel = Path("../data/fbm.db")
_abs = Path.home() / "Projects" / "fbm" / "data" / "fbm.db"
db_path = str(_rel if _rel.exists() else _abs)

conn = create_connection(db_path)

SEASONS = range(2019, 2025)
SOURCES = ["steamer", "playing_time", "actuals"]

In [ ]:
# Load data from all three sources across all seasons
data: dict[tuple[str, int], pd.DataFrame] = {}

for season in SEASONS:
    # Steamer projections
    df_steamer = pd.read_sql(
        "SELECT player_id, ip FROM projection WHERE system='steamer' AND player_type='pitcher' AND season=?",
        conn,
        params=[season],
    )
    if not df_steamer.empty:
        data[("steamer", season)] = df_steamer

    # Playing-time model
    df_pt = pd.read_sql(
        "SELECT player_id, ip FROM projection WHERE system='playing_time' AND player_type='pitcher' AND season=?",
        conn,
        params=[season],
    )
    if not df_pt.empty:
        data[("playing_time", season)] = df_pt

    # Actuals
    df_actual = pd.read_sql(
        "SELECT player_id, ip FROM pitching_stats WHERE source='fangraphs' AND season=?",
        conn,
        params=[season],
    )
    if not df_actual.empty:
        data[("actuals", season)] = df_actual

print(f"Loaded {len(data)} source×season combinations")
for (source, season), df in sorted(data.items()):
    print(f"  {source:15s} {season}: {len(df):5d} pitchers")

In [ ]:
def gini(values: np.ndarray) -> float:
    """Gini coefficient — 0 = perfect equality, 1 = maximum concentration."""
    v = np.sort(values)
    n = len(v)
    if n == 0 or v.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return float((2 * (index * v).sum() / (n * v.sum())) - (n + 1) / n)


rows = []
for (source, season), df in sorted(data.items()):
    ip = df["ip"].dropna().astype(float)
    rows.append(
        {
            "source": source,
            "season": season,
            "count_ip>0": int((ip > 0).sum()),
            "count_ip>30": int((ip > 30).sum()),
            "count_ip>60": int((ip > 60).sum()),
            "count_ip>100": int((ip > 100).sum()),
            "count_ip>150": int((ip > 150).sum()),
            "mean_ip": round(ip.mean(), 1),
            "median_ip": round(ip.median(), 1),
            "std_ip": round(ip.std(), 1),
            "gini": round(gini(ip.values), 3),
            "total_pool_ip": round(ip.sum(), 0),
        }
    )

metrics_df = pd.DataFrame(rows)
metrics_df.set_index(["source", "season"], inplace=True)
metrics_df

In [ ]:
percentiles = [10, 25, 50, 75, 90]
pct_rows = []
for (source, season), df in sorted(data.items()):
    ip = df["ip"].dropna().astype(float)
    row = {"source": source, "season": season}
    for p in percentiles:
        row[f"p{p}"] = round(float(np.percentile(ip, p)), 1)
    pct_rows.append(row)

pct_df = pd.DataFrame(pct_rows)
pct_df.set_index(["source", "season"], inplace=True)
pct_df

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
colors = {"steamer": "tab:blue", "playing_time": "tab:orange", "actuals": "tab:green"}
labels = {"steamer": "Steamer", "playing_time": "PT Model", "actuals": "Actuals"}

for i, season in enumerate(SEASONS):
    ax = axes[i // 3, i % 3]
    for source in SOURCES:
        key = (source, season)
        if key not in data:
            continue
        ip = np.sort(data[key]["ip"].dropna().astype(float).values)
        cdf = np.linspace(0, 1, len(ip))
        ax.plot(ip, cdf, label=labels[source], color=colors[source], linewidth=1.5)
    ax.set_title(f"{season}")
    ax.set_xlabel("IP")
    ax.set_ylabel("CDF")
    ax.set_xlim(0, 250)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Pitcher IP Distribution — CDF Overlay by Season", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compute target ranges from actual-season metrics (exclude 2020 COVID season)
actual_metrics = metrics_df.loc["actuals"].copy()
actual_full_seasons = actual_metrics[actual_metrics.index != 2020]

target_cols = [
    "count_ip>0",
    "count_ip>30",
    "count_ip>60",
    "count_ip>100",
    "count_ip>150",
    "mean_ip",
    "median_ip",
    "gini",
    "total_pool_ip",
]

target_rows = []
for col in target_cols:
    mean_val = actual_full_seasons[col].mean()
    std_val = actual_full_seasons[col].std()
    target_rows.append(
        {
            "metric": col,
            "actual_mean": round(mean_val, 1),
            "actual_std": round(std_val, 1),
            "target_low": round(mean_val - std_val, 1),
            "target_high": round(mean_val + std_val, 1),
        }
    )

targets_df = pd.DataFrame(target_rows).set_index("metric")
print("Target ranges for PT model calibration (actual mean ± 1 std, 2019-2024 excl. 2020):\n")
targets_df

## Target Summary

Target ranges are derived from full-season actuals (2019, 2021-2024), excluding the 2020 COVID-shortened season. Key takeaways:

- **Pitcher count (IP > 0):** Actuals have ~725-909 per season. Steamer projects ~1600-1850. The PT model currently projects ~1630-1880. Both need to be cut roughly in half to match reality.
- **High-IP starters (IP > 150):** Actuals have 55-73. Steamer (68-88) and PT model (45-67) are in the right ballpark already.
- **Mean IP:** Actuals average 46-54 IP. Steamer averages 25-33 (diluted by the long tail of 1 IP projections). PT model averages 38-45 — closer but still low.
- **Median IP:** Actuals are 32-42. Steamer median is always 1.0 (most pitchers projected for minimal IP). PT model median is 29-39, much more realistic.
- **Total pool IP:** Actuals total ~39,000-43,000. The PT model projects 67,000-74,000 — nearly double, due to projecting ~2× as many pitchers.
- **Gini coefficient:** Actuals are 0.50-0.53 (moderate concentration). Steamer is 0.68-0.75 (extreme — many pitchers with ~0 IP). PT model is 0.47-0.57, closer to actuals.

The primary calibration lever for phase 3 is **reducing the number of pitchers** who receive IP projections, while preserving the IP ranking and volume for those who do appear.